In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

import requests
import os

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
local_path = "/tmp/yellow_tripdata_2025-01.parquet"

# Download the Parquet file
with requests.get(url, stream=True, timeout=120) as response:
    response.raise_for_status()
    with open(local_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                file.write(chunk)

# Create the Bronze folder in the Lakehouse
notebookutils.fs.mkdirs("Files/bronze")

# Copy the downloaded file into the Lakehouse
notebookutils.fs.cp(
    "file:" + local_path,
    "Files/bronze/yellow_tripdata_2025-01.parquet"
)

print("File copied to Lakehouse.")

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 4, Finished, Available, Finished, False)

File copied to Lakehouse.


In [3]:

file_path = "Files/bronze/yellow_tripdata_2025-01.parquet"

df = ""

print(f"Number of columns: {len(df.columns)}")
print(f"Number of rows: {df.count()}")

df.printSchema()

display(df.limit(10))


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 5, Finished, Available, Finished, False)

Number of columns: 20
Number of rows: 3475226
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 972079a0-c561-428d-aa8f-6cef777113b6)

In [4]:
print("Column names:")
print(df.columns)

print("Data types:")
df.printSchema()

print("Missing values per column:")
from pyspark.sql import functions as F

df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 6, Finished, Available, Finished, False)

Column names:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Data types:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (null

In [5]:

from pyspark.sql import functions as F

df.filter(F.col("passenger_count").isNull()).select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee",
    "total_amount"
).show(10, truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 7, Finished, Available, Finished, False)

+--------+--------------------+---------------------+---------------+----------+------------------+--------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|RatecodeID|store_and_fwd_flag|congestion_surcharge|Airport_fee|total_amount|
+--------+--------------------+---------------------+---------------+----------+------------------+--------------------+-----------+------------+
|2       |2025-01-01 00:39:17 |2025-01-01 01:10:42  |NULL           |NULL      |NULL              |NULL                |NULL       |14.9        |
|2       |2025-01-01 00:33:40 |2025-01-01 01:10:34  |NULL           |NULL      |NULL              |NULL                |NULL       |10.02       |
|1       |2025-01-01 00:09:32 |2025-01-01 00:12:38  |NULL           |NULL      |NULL              |NULL                |NULL       |26.72       |
|2       |2025-01-01 00:21:42 |2025-01-01 00:36:29  |NULL           |NULL      |NULL              |NULL                |NULL

In [6]:

from pyspark.sql import functions as F

invalid_trips = df.filter(
    (F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")) |
    (F.col("trip_distance") < 0)
)

print(f"Invalid trips: {invalid_trips.count()}")

print(
    f"Negative distances: "
    f"{df.filter(F.col('trip_distance') < 0).count()}"
)

print(
    f"Dropoff before pickup: "
    f"{df.filter(F.col('tpep_dropoff_datetime') < F.col('tpep_pickup_datetime')).count()}"
)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 8, Finished, Available, Finished, False)

Invalid trips: 124
Negative distances: 0
Dropoff before pickup: 124


In [7]:

invalid_trips.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount"
).show(20, truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 9, Finished, Available, Finished, False)

+--------+--------------------+---------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|
+--------+--------------------+---------------------+-------------+-----------+------------+
|1       |2025-01-02 12:26:00 |2025-01-02 11:29:58  |9.0          |45.5       |46.0        |
|1       |2025-01-06 16:00:00 |2025-01-06 15:05:30  |3.8          |24.5       |25.0        |
|1       |2025-01-15 15:00:00 |2025-01-15 14:42:48  |1.0          |18.5       |19.0        |
|1       |2025-01-23 01:44:59 |2024-12-18 07:52:40  |3.1          |21.9       |30.4        |
|1       |2025-01-23 12:30:00 |2025-01-23 11:44:59  |3.9          |27.5       |28.0        |
|1       |2025-01-29 14:00:00 |2025-01-29 13:30:15  |1.5          |19.5       |20.0        |
|6       |2025-01-01 10:01:27 |2025-01-01 10:01:00  |19.19        |2.9        |68.77       |
|6       |2025-01-01 10:01:53 |2025-01-01 10:01:33  |12.17        |1.4

In [8]:

from pyspark.sql import functions as F

silver_df = df.filter(
    F.col("tpep_dropoff_datetime") >=
    F.col("tpep_pickup_datetime")
)

bronze_count = df.count()
silver_count = silver_df.count()
rejected_count = bronze_count - silver_count

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Rejected rows: {rejected_count:,}")
print(f"Rejected percentage: {rejected_count / bronze_count * 100:.4f}%")


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 10, Finished, Available, Finished, False)

Bronze rows: 3,475,226
Silver rows: 3,475,102
Rejected rows: 124
Rejected percentage: 0.0036%


In [9]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("yellow_taxi_silver")
)

print("Silver table created successfully.")

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 11, Finished, Available, Finished, False)

Silver table created successfully.


In [10]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM yellow_taxi_silver
""").show()

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 12, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|   3475102|
+----------+



In [11]:

from pyspark.sql import functions as F

columns_to_check = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

silver_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in columns_to_check
]).show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 13, Finished, Available, Finished, False)

+---------------+----------+------------------+--------------------+-----------+
|passenger_count|RatecodeID|store_and_fwd_flag|congestion_surcharge|Airport_fee|
+---------------+----------+------------------+--------------------+-----------+
|         540031|    540031|            540031|              540031|     540031|
+---------------+----------+------------------+--------------------+-----------+



In [12]:

from pyspark.sql import functions as F

silver_df.groupBy(
    *[
        F.col(c).isNull().alias(f"{c}_is_null")
        for c in [
            "passenger_count",
            "RatecodeID",
            "store_and_fwd_flag",
            "congestion_surcharge",
            "Airport_fee"
        ]
    ]
).count().orderBy(F.desc("count")).show(truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 14, Finished, Available, Finished, False)

+-----------------------+------------------+--------------------------+----------------------------+-------------------+-------+
|passenger_count_is_null|RatecodeID_is_null|store_and_fwd_flag_is_null|congestion_surcharge_is_null|Airport_fee_is_null|count  |
+-----------------------+------------------+--------------------------+----------------------------+-------------------+-------+
|false                  |false             |false                     |false                       |false              |2935071|
|true                   |true              |true                      |true                        |true               |540031 |
+-----------------------+------------------+--------------------------+----------------------------+-------------------+-------+



In [13]:

missing_values_df = silver_df.filter(
    F.col("passenger_count").isNull()
    & F.col("RatecodeID").isNull()
    & F.col("store_and_fwd_flag").isNull()
    & F.col("congestion_surcharge").isNull()
    & F.col("Airport_fee").isNull()
)

display(missing_values_df.limit(1000))


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 64d7167f-bb2f-47c9-b0a7-cb636e37f358)

In [14]:

from pyspark.sql import functions as F

silver_df.select(
    F.count("*").alias("total_trips"),
    F.sum(F.when(F.col("fare_amount") < 0, 1).otherwise(0)).alias("negative_fares"),
    F.sum(F.when(F.col("total_amount") < 0, 1).otherwise(0)).alias("negative_total_amounts"),
    F.sum(F.when(F.col("tip_amount") < 0, 1).otherwise(0)).alias("negative_tips"),
    F.sum(F.when(F.col("trip_distance") < 0, 1).otherwise(0)).alias("negative_distances"),
    F.sum(F.when(F.col("trip_distance") == 0, 1).otherwise(0)).alias("zero_distances"),
    F.sum(F.when(F.col("fare_amount").isNull(), 1).otherwise(0)).alias("null_fares"),
    F.sum(F.when(F.col("total_amount").isNull(), 1).otherwise(0)).alias("null_total_amounts")
).show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 16, Finished, Available, Finished, False)

+-----------+--------------+----------------------+-------------+------------------+--------------+----------+------------------+
|total_trips|negative_fares|negative_total_amounts|negative_tips|negative_distances|zero_distances|null_fares|null_total_amounts|
+-----------+--------------+----------------------+-------------+------------------+--------------+----------+------------------+
|    3475102|        144118|                 63037|          124|                 0|         90893|         0|                 0|
+-----------+--------------+----------------------+-------------+------------------+--------------+----------+------------------+



In [15]:

silver_df.filter(
    (F.col("fare_amount") < 0) |
    (F.col("total_amount") < 0) |
    (F.col("trip_distance") == 0)
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",
    "VendorID"
).show(30, truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 17, Finished, Available, Finished, False)

+--------------------+---------------------+-------------+-----------+----------+------------+------------+------------+--------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|tip_amount|tolls_amount|total_amount|payment_type|VendorID|
+--------------------+---------------------+-------------+-----------+----------+------------+------------+------------+--------+
|2025-01-01 00:01:41 |2025-01-01 00:07:14  |0.71         |-7.2       |3.66      |0.0         |-8.54       |2           |2       |
|2025-01-01 00:55:54 |2025-01-01 01:00:38  |0.69         |-6.5       |0.0       |0.0         |-11.5       |4           |2       |
|2025-01-01 00:49:48 |2025-01-01 00:49:48  |0.0          |20.06      |0.0       |0.0         |20.06       |2           |1       |
|2025-01-01 00:56:12 |2025-01-01 01:15:00  |0.97         |-16.3      |0.0       |0.0         |-21.3       |4           |2       |
|2025-01-01 00:55:53 |2025-01-01 01:06:49  |1.42         |-12.1      |0.0       |0.0      

In [16]:

duration_minutes = (
    F.unix_timestamp("tpep_dropoff_datetime")
    - F.unix_timestamp("tpep_pickup_datetime")
) / 60

silver_df.select(
    F.sum(
        F.when(duration_minutes == 0, 1).otherwise(0)
    ).alias("zero_duration_trips"),
    F.sum(
        F.when(duration_minutes > 180, 1).otherwise(0)
    ).alias("trips_over_3_hours"),
    F.round(F.min(duration_minutes), 2).alias("min_duration_minutes"),
    F.round(F.expr(
        "percentile_approx((unix_timestamp(tpep_dropoff_datetime) - "
        "unix_timestamp(tpep_pickup_datetime)) / 60, 0.5)"
    ), 2).alias("median_duration_minutes"),
    F.round(F.max(duration_minutes), 2).alias("max_duration_minutes")
).show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 18, Finished, Available, Finished, False)

+-------------------+------------------+--------------------+-----------------------+--------------------+
|zero_duration_trips|trips_over_3_hours|min_duration_minutes|median_duration_minutes|max_duration_minutes|
+-------------------+------------------+--------------------+-----------------------+--------------------+
|               1927|              1377|                 0.0|                   11.7|             5626.32|
+-------------------+------------------+--------------------+-----------------------+--------------------+



In [17]:

silver_df.withColumn(
    "duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
).orderBy(
    F.desc("duration_minutes")
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "duration_minutes",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "VendorID",
    "payment_type"
).show(20, truncate=False)



StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 19, Finished, Available, Finished, False)

+--------------------+---------------------+------------------+-------------+-----------+------------+--------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|duration_minutes  |trip_distance|fare_amount|total_amount|VendorID|payment_type|
+--------------------+---------------------+------------------+-------------+-----------+------------+--------+------------+
|2025-01-21 12:23:41 |2025-01-25 10:10:00  |5626.316666666667 |6.55         |-28.2      |-36.45      |2       |2           |
|2025-01-21 12:23:41 |2025-01-25 10:10:00  |5626.316666666667 |6.55         |28.2       |36.45       |2       |2           |
|2025-01-06 02:12:11 |2025-01-09 13:54:19  |5022.133333333333 |91.51        |336.9      |339.4       |2       |2           |
|2025-01-27 11:58:55 |2025-01-30 09:42:55  |4184.0            |0.0          |3.0        |4.5         |2       |2           |
|2025-01-11 08:04:00 |2025-01-13 16:49:28  |3405.4666666666667|0.04         |70.0       |76.5        |2       |2           |


In [18]:

from pyspark.sql import functions as F

total_rows = silver_df.count()
distinct_rows = silver_df.distinct().count()

print(f"Total rows: {total_rows:,}")
print(f"Distinct rows: {distinct_rows:,}")
print(f"Exact duplicate rows: {total_rows - distinct_rows:,}")


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 20, Finished, Available, Finished, False)

Total rows: 3,475,102
Distinct rows: 3,475,102
Exact duplicate rows: 0


In [19]:

trip_groups = (
    silver_df
    .groupBy(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "VendorID"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.col("fare_amount") < 0, 1).otherwise(0)
        ).alias("negative_fares"),
        F.sum(
            F.when(F.col("fare_amount") > 0, 1).otherwise(0)
        ).alias("positive_fares")
    )
)

opposite_fare_groups = trip_groups.filter(
    (F.col("negative_fares") > 0) &
    (F.col("positive_fares") > 0)
)

print(
    "Groups with both negative and positive fares:",
    opposite_fare_groups.count()
)

opposite_fare_groups.orderBy(
    F.desc("row_count")
).show(20, truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 21, Finished, Available, Finished, False)

Groups with both negative and positive fares: 59294
+--------------------+---------------------+-------------+--------+---------+--------------+--------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|VendorID|row_count|negative_fares|positive_fares|
+--------------------+---------------------+-------------+--------+---------+--------------+--------------+
|2025-01-10 12:35:38 |2025-01-10 12:38:27  |0.49         |2       |3        |1             |2             |
|2025-01-12 15:22:09 |2025-01-12 15:22:13  |0.0          |2       |3        |1             |2             |
|2025-01-10 17:53:21 |2025-01-10 18:00:16  |1.14         |2       |3        |1             |2             |
|2025-01-01 01:46:11 |2025-01-01 02:06:23  |3.35         |2       |2        |1             |1             |
|2025-01-01 00:12:46 |2025-01-01 00:18:14  |0.8          |2       |2        |1             |1             |
|2025-01-01 06:09:18 |2025-01-01 06:12:52  |0.73         |2       |2        |1      

In [20]:

paired_groups = (
    silver_df
    .groupBy(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "VendorID"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.sum("fare_amount").alias("net_fare"),
        F.sum("total_amount").alias("net_total"),
        F.min("fare_amount").alias("min_fare"),
        F.max("fare_amount").alias("max_fare"),
        F.min("total_amount").alias("min_total"),
        F.max("total_amount").alias("max_total")
    )
    .filter(F.col("row_count") == 2)
    .filter(
        (F.col("min_fare") < 0) &
        (F.col("max_fare") > 0)
    )
)

print("Opposite-sign pairs:", paired_groups.count())

paired_groups.orderBy(
    F.abs(F.col("net_total")).asc()
).show(20, truncate=False)


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 22, Finished, Available, Finished, False)

Opposite-sign pairs: 59291
+--------------------+---------------------+-------------+--------+---------+--------+---------+--------+--------+---------+---------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|VendorID|row_count|net_fare|net_total|min_fare|max_fare|min_total|max_total|
+--------------------+---------------------+-------------+--------+---------+--------+---------+--------+--------+---------+---------+
|2025-01-03 14:22:58 |2025-01-03 14:32:58  |1.99         |2       |2        |0.0     |0.0      |-12.8   |12.8    |-16.8    |16.8     |
|2025-01-01 00:12:46 |2025-01-01 00:18:14  |0.8          |2       |2        |0.0     |0.0      |-7.2    |7.2     |-12.2    |12.2     |
|2025-01-02 21:45:18 |2025-01-02 21:58:41  |2.39         |2       |2        |0.0     |0.0      |-14.9   |14.9    |-19.9    |19.9     |
|2025-01-01 00:34:05 |2025-01-01 01:16:34  |16.1         |2       |2        |0.0     |0.0      |-67.4   |67.4    |-72.4    |72.4     |
|2025-01-02 21:47:14 |2025-0

In [21]:

silver_df.agg(
    F.round(F.sum("fare_amount"), 2).alias("net_fare_amount"),
    F.round(F.sum("total_amount"), 2).alias("net_total_amount"),
    F.round(
        F.sum(
            F.when(F.col("fare_amount") < 0, F.col("fare_amount"))
            .otherwise(0)
        ), 2
    ).alias("sum_negative_fares"),
    F.round(
        F.sum(
            F.when(F.col("total_amount") < 0, F.col("total_amount"))
            .otherwise(0)
        ), 2
    ).alias("sum_negative_total_amounts")
).show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 23, Finished, Available, Finished, False)

+---------------+----------------+------------------+--------------------------+
|net_fare_amount|net_total_amount|sum_negative_fares|sum_negative_total_amounts|
+---------------+----------------+------------------+--------------------------+
|  5.936261678E7|   8.900191717E7|       -1676226.41|               -1657489.57|
+---------------+----------------+------------------+--------------------------+



In [22]:

# Final validation: Bronze -> Silver

print(f"Bronze DataFrame: {bronze_count:,}")
print(f"Silver DataFrame: {silver_count:,}")
print(f"Rejected rows: {rejected_count:,}")

print("Silver Delta table:")
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM yellow_taxi_silver
""").show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 24, Finished, Available, Finished, False)

Bronze DataFrame: 3,475,226
Silver DataFrame: 3,475,102
Rejected rows: 124
Silver Delta table:
+----------+
|total_rows|
+----------+
|   3475102|
+----------+



In [23]:

silver_df.select(
    F.countDistinct("VendorID").alias("distinct_vendors"),
    F.countDistinct("PULocationID").alias("pickup_zones"),
    F.countDistinct("DOLocationID").alias("dropoff_zones"),
    F.countDistinct("payment_type").alias("payment_types"),
    F.sum(
        F.when(F.col("VendorID").isNull(), 1).otherwise(0)
    ).alias("null_vendors"),
    F.sum(
        F.when(F.col("PULocationID").isNull(), 1).otherwise(0)
    ).alias("null_pickup_zones"),
    F.sum(
        F.when(F.col("DOLocationID").isNull(), 1).otherwise(0)
    ).alias("null_dropoff_zones"),
    F.sum(
        F.when(F.col("payment_type").isNull(), 1).otherwise(0)
    ).alias("null_payment_types")
).show()


StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 25, Finished, Available, Finished, False)

+----------------+------------+-------------+-------------+------------+-----------------+------------------+------------------+
|distinct_vendors|pickup_zones|dropoff_zones|payment_types|null_vendors|null_pickup_zones|null_dropoff_zones|null_payment_types|
+----------------+------------+-------------+-------------+------------+-----------------+------------------+------------------+
|               4|         261|          260|            6|           0|                0|                 0|                 0|
+----------------+------------+-------------+-------------+------------+-----------------+------------------+------------------+



In [24]:
import requests

url = "https://s3.amazonaws.com/nyc-tlc/misc/taxi+zone_lookup.csv"

response = requests.get(url)
response.raise_for_status()

mssparkutils.fs.mkdirs("Files/reference")

mssparkutils.fs.put(
    "Files/reference/taxi_zone_lookup.csv",
    response.text,
    True
)

print("CSV downloaded:", len(response.text), "characters")

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 26, Finished, Available, Finished, False)

HTTPError: 403 Client Error: Forbidden for url: https://s3.amazonaws.com/nyc-tlc/misc/taxi+zone_lookup.csv

In [26]:
import requests

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(url, timeout=60)
response.raise_for_status()

print(response.text[:500])

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 28, Finished, Available, Finished, False)

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"
10,"Queens","Baisley Park","Boro Zone"
11,"Brooklyn","Bath Beach","Boro Zon


In [27]:
mssparkutils.fs.mkdirs("Files/reference")

mssparkutils.fs.put(
    "Files/reference/taxi_zone_lookup.csv",
    response.text,
    True
)

print("Taxi Zone Lookup saved successfully.")

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 29, Finished, Available, Finished, False)

Taxi Zone Lookup saved successfully.


In [28]:
from pyspark.sql import functions as F

zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/taxi_zone_lookup.csv")
)

print("Rows in lookup:", zones_df.count())
zones_df.printSchema()

pickup_ids = silver_df.select(
    F.col("PULocationID").alias("LocationID")
)

dropoff_ids = silver_df.select(
    F.col("DOLocationID").alias("LocationID")
)

used_ids = pickup_ids.union(dropoff_ids).distinct()

missing_ids = used_ids.join(
    zones_df.select("LocationID").distinct(),
    on="LocationID",
    how="left_anti"
)

print("Distinct used location IDs:", used_ids.count())
print("IDs missing from lookup:", missing_ids.count())

missing_ids.orderBy("LocationID").show(100, truncate=False)

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 30, Finished, Available, Finished, False)

Rows in lookup: 265
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Distinct used location IDs: 262
IDs missing from lookup: 0
+----------+
|LocationID|
+----------+
+----------+



In [29]:
dim_taxi_zone = (
    zones_df
    .select(
        F.col("LocationID").alias("LocationID"),
        F.col("Borough").alias("Borough"),
        F.col("Zone").alias("Zone"),
        F.col("service_zone").alias("ServiceZone")
    )
    .dropDuplicates(["LocationID"])
)

(dim_taxi_zone.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimTaxiZone")
)

spark.sql("""
    SELECT *
    FROM DimTaxiZone
    ORDER BY LocationID
    LIMIT 10
""").show(truncate=False)

print("DimTaxiZone rows:", spark.table("DimTaxiZone").count())

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 31, Finished, Available, Finished, False)

+----------+-------------+-----------------------+-----------+
|LocationID|Borough      |Zone                   |ServiceZone|
+----------+-------------+-----------------------+-----------+
|1         |EWR          |Newark Airport         |EWR        |
|2         |Queens       |Jamaica Bay            |Boro Zone  |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone  |
|4         |Manhattan    |Alphabet City          |Yellow Zone|
|5         |Staten Island|Arden Heights          |Boro Zone  |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone  |
|7         |Queens       |Astoria                |Boro Zone  |
|8         |Queens       |Astoria Park           |Boro Zone  |
|9         |Queens       |Auburndale             |Boro Zone  |
|10        |Queens       |Baisley Park           |Boro Zone  |
+----------+-------------+-----------------------+-----------+

DimTaxiZone rows: 265


In [30]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM DimTaxiZone
""").show()

StatementMeta(, 4a99ea3f-3021-457c-9d37-d6832eac04f2, 32, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|       265|
+----------+



In [2]:
from pyspark.sql import functions as F

silver_df = spark.read.table("yellow_taxi_silver")

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))

StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 4, Finished, Available, Finished, False)

Rows: 3475102
Columns: 20


In [3]:
silver_df.printSchema()

StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 5, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:

from pyspark.sql import functions as F

fact_taxi_trips = (
    silver_df
    .withColumn(
        "PickupDateKey",
        F.date_format("tpep_pickup_datetime", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "DropoffDateKey",
        F.date_format("tpep_dropoff_datetime", "yyyyMMdd").cast("int")
    )
    .select(
        "VendorID",
        "PickupDateKey",
        "DropoffDateKey",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "payment_type",
        "passenger_count",
        "trip_distance",
        "RatecodeID",
        "store_and_fwd_flag",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "Airport_fee",
        "cbd_congestion_fee"
    )
)

fact_taxi_trips.printSchema()
print("Fact rows:", fact_taxi_trips.count())


StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 6, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- PickupDateKey: integer (nullable = true)
 |-- DropoffDateKey: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (

In [5]:
(
    fact_taxi_trips.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("FactTaxiTrips")
)

spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM FactTaxiTrips
""").show()

StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 7, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|   3475102|
+----------+



In [6]:

from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiTrips")
dim_vendor = spark.read.table("DimVendor")
dim_payment = spark.read.table("DimPaymentType")
dim_zone = spark.read.table("DimTaxiZone")
dim_date = spark.read.table("DimDate")

fact.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("VendorID").isNull(), 1).otherwise(0)).alias("null_vendor"),
    F.sum(F.when(F.col("payment_type").isNull(), 1).otherwise(0)).alias("null_payment_type"),
    F.sum(F.when(F.col("PULocationID").isNull(), 1).otherwise(0)).alias("null_pickup_zone"),
    F.sum(F.when(F.col("DOLocationID").isNull(), 1).otherwise(0)).alias("null_dropoff_zone"),
    F.sum(F.when(F.col("PickupDateKey").isNull(), 1).otherwise(0)).alias("null_pickup_date"),
    F.sum(F.when(F.col("DropoffDateKey").isNull(), 1).otherwise(0)).alias("null_dropoff_date")
).show()


StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 8, Finished, Available, Finished, False)

+----------+-----------+-----------------+----------------+-----------------+----------------+-----------------+
|total_rows|null_vendor|null_payment_type|null_pickup_zone|null_dropoff_zone|null_pickup_date|null_dropoff_date|
+----------+-----------+-----------------+----------------+-----------------+----------------+-----------------+
|   3475102|          0|                0|               0|                0|               0|                0|
+----------+-----------+-----------------+----------------+-----------------+----------------+-----------------+



In [7]:

from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiTrips")

checks = [
    ("VendorID", "DimVendor", "VendorID"),
    ("payment_type", "DimPaymentType", "payment_type"),
    ("PULocationID", "DimTaxiZone", "LocationID"),
    ("DOLocationID", "DimTaxiZone", "LocationID"),
    ("PickupDateKey", "DimDate", "Date"),
    ("DropoffDateKey", "DimDate", "Date")
]

for fact_col, dim_table, dim_col in checks:
    dim = spark.read.table(dim_table)

    if "Date" in dim.columns and dim_col == "Date":
        dim_keys = dim.select(
            F.date_format(F.col("Date"), "yyyyMMdd")
             .cast("int")
             .alias("dim_key")
        )
    else:
        dim_keys = dim.select(
            F.col(dim_col).alias("dim_key")
        )

    missing = (
        fact.select(F.col(fact_col).alias("fact_key"))
        .distinct()
        .join(
            dim_keys.distinct(),
            F.col("fact_key") == F.col("dim_key"),
            "left_anti"
        )
    )

    print(
        f"{fact_col} -> {dim_table}: "
        f"{missing.count()} unmatched keys"
    )


StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 9, Finished, Available, Finished, False)

VendorID -> DimVendor: 0 unmatched keys
payment_type -> DimPaymentType: 0 unmatched keys
PULocationID -> DimTaxiZone: 0 unmatched keys
DOLocationID -> DimTaxiZone: 0 unmatched keys
PickupDateKey -> DimDate: 1 unmatched keys
DropoffDateKey -> DimDate: 1 unmatched keys


In [8]:

from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiTrips")
dim_date = spark.read.table("DimDate")

date_keys = dim_date.select(
    F.date_format("Date", "yyyyMMdd")
     .cast("int")
     .alias("DateKey")
).distinct()

for key_col, timestamp_col in [
    ("PickupDateKey", "tpep_pickup_datetime"),
    ("DropoffDateKey", "tpep_dropoff_datetime")
]:
    missing_keys = (
        fact.select(key_col)
        .distinct()
        .join(
            date_keys,
            F.col(key_col) == F.col("DateKey"),
            "left_anti"
        )
    )

    print(f"\nMissing keys for {key_col}:")
    missing_keys.show(truncate=False)

    print(f"Example rows for {key_col}:")
    fact.join(
        missing_keys,
        fact[key_col] == missing_keys[key_col],
        "inner"
    ).select(
        key_col,
        timestamp_col,
        "VendorID",
        "trip_distance",
        "fare_amount",
        "total_amount"
    ).show(10, truncate=False)


StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 10, Finished, Available, Finished, False)


Missing keys for PickupDateKey:
+-------------+
|PickupDateKey|
+-------------+
|20241231     |
+-------------+

Example rows for PickupDateKey:


AnalysisException: [AMBIGUOUS_REFERENCE] Reference `PickupDateKey` is ambiguous, could be: [`spark_catalog`.`chimcobldhq2ajj9cdqiqh31ehgiqhbectkmspb5e9kmsppd9hgm49bcd1fmsub3btq62u394li64ro`.`FactTaxiTrips`.`PickupDateKey`, `spark_catalog`.`chimcobldhq2ajj9cdqiqh31ehgiqhbectkmspb5e9kmsppd9hgm49bcd1fmsub3btq62u394li64ro`.`FactTaxiTrips`.`PickupDateKey`]. SQLSTATE: 42704

In [9]:
fact = spark.read.table("FactTaxiTrips")

fact.filter(
    (F.col("PickupDateKey") == 20241231) |
    (F.col("DropoffDateKey") == 20241231)
).select(
    "PickupDateKey",
    "DropoffDateKey",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(20, truncate=False)

StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 11, Finished, Available, Finished, False)

+-------------+--------------+--------------------+---------------------+
|PickupDateKey|DropoffDateKey|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------+--------------+--------------------+---------------------+
|20241231     |20241231      |2024-12-31 23:30:03 |2024-12-31 23:43:02  |
|20241231     |20241231      |2024-12-31 23:31:38 |2024-12-31 23:41:48  |
|20241231     |20250101      |2024-12-31 23:46:38 |2025-01-01 00:03:03  |
|20241231     |20250101      |2024-12-31 23:56:19 |2025-01-01 00:11:19  |
|20241231     |20250101      |2024-12-31 23:55:37 |2025-01-01 00:01:26  |
|20241231     |20250101      |2024-12-31 23:52:40 |2025-01-01 00:23:03  |
|20241231     |20241231      |2024-12-31 23:49:24 |2024-12-31 23:57:30  |
|20241231     |20241231      |2024-12-31 23:27:13 |2024-12-31 23:35:48  |
|20241231     |20241231      |2024-12-31 23:37:42 |2024-12-31 23:43:10  |
|20241231     |20250101      |2024-12-31 23:51:20 |2025-01-01 00:00:00  |
|20241231     |20250101      |2024-12-

In [10]:

from pyspark.sql import functions as F

fact = spark.read.table("FactTaxiTrips")
dim_vendor = spark.read.table("DimVendor")
dim_payment = spark.read.table("DimPaymentType")
dim_zone = spark.read.table("DimTaxiZone")
dim_date = spark.read.table("DimDate")

checks = [
    ("VendorID", fact, "VendorID", dim_vendor, "VendorID"),
    ("payment_type", fact, "payment_type", dim_payment, "payment_type"),
    ("PULocationID", fact, "PULocationID", dim_zone, "LocationID"),
    ("DOLocationID", fact, "DOLocationID", dim_zone, "LocationID"),
    ("PickupDateKey", fact, "PickupDateKey", dim_date, "DateKey"),
    ("DropoffDateKey", fact, "DropoffDateKey", dim_date, "DateKey")
]

for name, fact_df, fact_col, dim_df, dim_col in checks:
    unmatched = (
        fact_df.select(F.col(fact_col).alias("key"))
        .distinct()
        .join(
            dim_df.select(F.col(dim_col).alias("key")).distinct(),
            on="key",
            how="left_anti"
        )
        .count()
    )
    print(f"{name} -> unmatched keys: {unmatched}")


StatementMeta(, c7e5597a-e3e9-4f14-bf39-f6434697eafa, 12, Finished, Available, Finished, False)

VendorID -> unmatched keys: 0
payment_type -> unmatched keys: 0
PULocationID -> unmatched keys: 0
DOLocationID -> unmatched keys: 0
PickupDateKey -> unmatched keys: 0
DropoffDateKey -> unmatched keys: 0


In [1]:
(
    spark.read.table("FactTaxiTrips")
    .groupBy("payment_type")
    .count()
    .orderBy("payment_type")
    .show()
)

StatementMeta(, 58eb0df5-d0b6-40a4-b5e8-3ddb9a6d51ce, 3, Finished, Available, Finished, False)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 540031|
|           1|2444387|
|           2| 390429|
|           3|  23773|
|           4|  76481|
|           5|      1|
+------------+-------+



In [3]:

from pyspark.sql import functions as F

payment_types = [
    (0, "Unspecified / Missing", "Unclassified", False),
    (1, "Credit card", "Electronic", True),
    (2, "Cash", "Cash", True),
    (3, "No charge", "Other", True),
    (4, "Dispute", "Other", True),
    (5, "Unknown", "Unclassified", False)
]

dim_payment_type = spark.createDataFrame(
    payment_types,
    [
        "payment_type",
        "PaymentTypeName",
        "PaymentCategory",
        "IsStandardPayment"
    ]
)

(
    dim_payment_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("DimPaymentType")
)

display(
    spark.read.table("DimPaymentType")
    .orderBy("payment_type")
)


StatementMeta(, 58eb0df5-d0b6-40a4-b5e8-3ddb9a6d51ce, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 127c06e5-0bb9-4951-8ee4-34a3cd2809e4)